<br><br>
<div style="text-align: center;">

<h1 style="
  margin: 0 0 18px 0;
  color: #0077c8;
  font-size: 42px;
  letter-spacing: 1px;
">
  TB Lightning Home Assessment
</h1>
<p style="
  margin: 0 0 20px 0;
  color: #4a5b68;
  font-size: 16px;
  text-align: center;
  width: 100%;
">
  <strong>Created by Tyler Durette</strong>
  &nbsp;·&nbsp;
</p>
    
  <div style="
    height: 4px;
    width: 42%;
    margin: 0 auto 22px auto;
    background: linear-gradient(90deg, #0077c8, #79c2ed, #ffb81c);
    border-radius: 4px;
  "></div>

  <img
    src="images/vasilevskiy.jpeg"
    alt="Andrei Vasilevskiy, Tampa Bay Lightning"
    style="
      display: block;
      width: 88%;
      max-width: 960px;
      margin: 0 auto;
      border-radius: 8px;
      box-shadow: 0 8px 24px rgba(0,55,95,.18);
    "
  />

  <p style="margin: 14px 0 0 0; color: #687985; font-size: 11px;  text-align: center">
    Andrei Vasilevskiy · Tampa Bay Lightning · image source: NHL.com
  </p>

</div>

<div style="
  border-left: 5px solid #ffb81c;
  padding: 16px 22px;
  background: #fffaf0;
  border-radius: 6px;
  color: #12304a;
">

  <h2 style="
    margin: 0 0 14px 0;
    color: #12304a;
    font-size: 24px;
  ">Overview</h2>

  <p style="
    margin: 0 0 14px 0;
    color: #4a5b68;
  ">This notebook is a visually appealing way walkthrough the NHL take-home implementation finished product I came up with as of now, including the required data import, cleaning, database modeling, API layer, robustness checks, and reproducible Docker workflow. <strong><i>There are many improvements or areas that could be optimized, but given the short timeframe of this task, functionality and data correctness was priority first.</strong></i></p>

  <ul style="
    margin: 0;
    padding-left: 22px;
    color: #4a5b68;
    line-height: 1.7;
  ">
    <li><strong>Data ingestion:</strong> Imports and validates the supplied 2022–23 seed dataset containing 951 skater records.</li>
    <li><strong>Historical coverage:</strong> Loads every contiguous regular season from 2022–23 through the configured 2026–27 target without skipping intermediate years.</li>
    <li><strong>Data modeling:</strong> Normalizes players, season statistics, games, team-game statistics, standings, rosters, and pipeline-run metadata.</li>
    <li><strong>Database:</strong> Uses PostgreSQL through Docker Compose locally and Supabase PostgreSQL as the managed cloud database.</li>
    <li><strong>API:</strong> Exposes FastAPI endpoints for player goals, penalty rates, team rankings, multi-team players, current rosters, health, and pipeline status.</li>
    <li><strong>Daily refresh:</strong> Using a Scheduled GitHub Action it rechecks the previous three dates, refreshes current-season statistics, standings, rosters, and correction-window game data every morning at 8am.</li>
    <li><strong>Robustness:</strong> Includes retries, pagination validation, preseason empty-state handling, idempotent upserts, season-gap validation, and pipeline status tracking.</li>
    <li><strong>Reproducibility:</strong> Runs through Docker Compose with documented commands, automated tests, and a GitHub Actions workflow for the managed PostgreSQL refresh.</li>
  </ul>

</div>

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import os
import re
import pandas as pd
import requests
from IPython.display import display, HTML, Image as NotebookImage

# PREREQUISITES

***MUST HAVE DOCKER INSTALLED AND THE CONTAINERS STARTED BEFORE RUNNING REST OF THE NOTEBOOK:***

```bash
# From the repository root
> ./run_docker_compose.sh status # Check the Docker service status.
> ./run_docker_compose.sh start # Start the services if they are stopped.
> curl -fsS http://localhost:8000/health | python -m json.tool # Confirm that FastAPI can reach PostgreSQL.
```

---

# DATA AND API QUALITY CHECKS

Basic dataset checks and initial exploratory data analysis before moving onto testing more advanced endpoint calls and calculations

In [ ]:
# Verify that the local API is reachable before running analysis cells.

!curl http://localhost:8000/health

In [ ]:
# Display the shared API configuration and notebook execution timestamp.

BASE_URL = os.environ.get("NHL_API_URL", "http://localhost:8000")
SEED_SEASON = 20222023
REQUEST_TIMEOUT = 10

print({
    "api_base_url": BASE_URL,
    "checked_at_utc": (
        datetime.now(timezone.utc).isoformat(timespec="seconds") + "Z"
    ),
})

In [ ]:
# Dataset shape, distributions, overview

seed_path = Path("data/data_dump.csv")
seed = pd.read_csv(seed_path, keep_default_na=False)
goals = pd.to_numeric(seed["G"], errors="coerce")
assists = pd.to_numeric(seed["A"], errors="coerce")
points = pd.to_numeric(seed["P"], errors="coerce")
shots = pd.to_numeric(seed["S"], errors="coerce")
shooting_pct = pd.to_numeric(seed["S%"], errors="coerce")

unique_teams = (
    seed["Team"]
    .astype(str)
    .str.split(",")
    .explode()
    .str.strip()
    .nunique()
)

# Shooting percentage is only expected to be populated when shots > 0
shooting_pct_valid = (
    (shots == 0) & shooting_pct.isna()
) | (
    (shots > 0)
    & shooting_pct.notna()
    & ((shooting_pct - (goals / shots)).abs() < 1e-4)
)

quality_summary = pd.DataFrame(
    {
        "check": [
            "Rows",
            "Columns",
            "Unique player IDs",
            "Season values",
            "Unique teams",
            "Multi-team player rows",
            "Null shooting percentages",
            "Null faceoff percentages",
            "P = G + A validation",
            "Shooting percentage validation",
        ],
        "result": [
            len(seed),
            len(seed.columns),
            seed["playerId"].nunique(),
            ", ".join(seed["Season"].astype(str).unique()),
            unique_teams,
            int(seed["Team"].str.contains(",", regex=False).sum()),
            int(shooting_pct.isna().sum()),
            int((seed["FOW%"].astype(str) == "None").sum()),
            bool((points == goals + assists).all()),
            bool(shooting_pct_valid.all()),
        ],
    }
)

display(quality_summary)

In [ ]:
# Full column-level profile

profile_rows = []
for column in seed.columns:
    series = seed[column]
    cleaned = series.replace({"": pd.NA, "None": pd.NA})
    numeric = pd.to_numeric(cleaned, errors="coerce")

    is_numeric = numeric.notna().sum() >= max(1, int(len(series) * 0.8))

    if is_numeric:
        non_null = numeric.dropna()
        summary = {
            "column": column,
            "dtype": str(series.dtype),
            "non_null": int(non_null.count()),
            "null_or_empty": int(len(series) - non_null.count()),
            "unique_values": int(series.nunique(dropna=True)),
            "min": non_null.min() if not non_null.empty else None,
            "max": non_null.max() if not non_null.empty else None,
            "mean": round(float(non_null.mean()), 3)
            if not non_null.empty
            else None,
            "example_values": ", ".join(
                str(value) for value in series.drop_duplicates().head(5)
            ),
        }
    else:
        value_counts = (
            series.astype(str)
            .replace({"": "NULL", "None": "NULL"})
            .value_counts()
            .head(5)
        )

        examples = "; ".join(
            f"{value} ({count})"
            for value, count in value_counts.items()
        )

        summary = {
            "column": column,
            "dtype": str(series.dtype),
            "non_null": int(series.replace({"": pd.NA, "None": pd.NA}).notna().sum()),
            "null_or_empty": int(
                series.replace({"": pd.NA, "None": pd.NA}).isna().sum()
            ),
            "unique_values": int(series.nunique(dropna=True)),
            "min": None,
            "max": None,
            "mean": None,
            "example_values": examples,
        }

    profile_rows.append(summary)

column_profile = pd.DataFrame(profile_rows)
display(column_profile)

In [ ]:
# API contract check

openapi = requests.get(
    f"{BASE_URL}/openapi.json",
    timeout=REQUEST_TIMEOUT,
)

openapi.raise_for_status()

expected_routes = [
    "/health",
    "/pipeline/status",
    "/players/most-goals",
    "/players/penalties-per-minute",
    "/players/leaderboard",
    "/players/multi-team",
    "/teams/rankings",
    "/standings",
    "/rosters/current/{team_abbrev}",
]

available_routes = sorted(openapi.json()["paths"])

route_check = pd.DataFrame(
    {
        "route": expected_routes,
        "available": [
            route in available_routes
            for route in expected_routes
        ],
    }
)

display(route_check)

assert route_check["available"].all()
print("API contract check passed.")

---

# ENDPOINT TESTS & EXAMPLES
The cells below mirror `tests/test_endpoints.sh` in the same order. Each response is rendered as a compact table or list so the Python client behavior is easy to inspect.

## 1. Health check

Confirm that the API is running and connected to its database.

In [ ]:
# Check API health and database reachability.

BASE_URL = os.environ.get("NHL_API_URL", "http://localhost:8000")
SEED_SEASON = 20222023
REQUEST_TIMEOUT = 10

def get_json(path, **params):
    response = requests.get(
        f"{BASE_URL}{path}",
        params=params,
        timeout=REQUEST_TIMEOUT,
    )
    response.raise_for_status()
    return response.json()

health = get_json('/health')
display(pd.Series(health, name='value').to_frame())

## 2. Top goal scorer — 2022–23

The supplied season's leading scorer should be Connor McDavid with 64 goals.

In [ ]:
# Identify and validate the leading goal scorer for the seed season.

goal_leader = get_json('/players/most-goals', season_id=SEED_SEASON)
goal_table = pd.DataFrame(goal_leader)
display(goal_table)
assert goal_table.iloc[0]['name'] == 'Connor McDavid'
assert int(goal_table.iloc[0]['goals']) == 64
print('Acceptance check passed: Connor McDavid led 2022–23 with 64 goals.')

## 3. Penalty minutes per total ice-time minute — 2022–23

This matches the shell test's minimum-50-game filter.

In [ ]:
# Rank players by penalty minutes per total ice-time minute.

penalties = get_json(
    '/players/penalties-per-minute',
    season_id=SEED_SEASON,
    min_games=50,
    limit=5,
)
penalty_table = pd.DataFrame(penalties)
display(penalty_table)
assert {'pim', 'total_toi_minutes', 'pim_per_minute'}.issubset(penalty_table.columns)
assert (penalty_table['total_toi_minutes'] > 0).all()
print(f'Validated {len(penalty_table)} penalty-rate rows with positive total TOI.')

## 4. Top player points — 2022–23

In [ ]:
# Display the leading players by total points.

points = get_json('/players/leaderboard', season_id=SEED_SEASON, metric='points', limit=5)
points_table = pd.DataFrame(points)
display(points_table)

## 5. Top player assists — 2022–23

In [ ]:
# Display the leading players by total assists.

assists = get_json('/players/leaderboard', season_id=SEED_SEASON, metric='assists', limit=5)
assists_table = pd.DataFrame(assists)
display(assists_table)

## 6. Top shooting percentage — 2022–23

The shell test applies a minimum-50-game sample filter.

In [ ]:
# Display the leading qualified players by shooting percentage.

shooting = get_json(
    '/players/leaderboard',
    season_id=SEED_SEASON,
    metric='shooting_pct',
    min_games=50,
    limit=5,
)
shooting_table = pd.DataFrame(shooting)
display(shooting_table)

## 7. Top teams by goals — 2022–23

In [ ]:
# Rank teams by goals scored.

team_goals = get_json('/teams/rankings', season_id=SEED_SEASON, metric='goals', limit=5)
team_goals_table = pd.DataFrame(team_goals)
display(team_goals_table)
assert team_goals_table['goals'].tolist() == sorted(team_goals_table['goals'], reverse=True)

## 8. Top teams by shots — 2022–23

In [ ]:
# Rank teams by shots taken and validate descending order.

team_shots = get_json('/teams/rankings', season_id=SEED_SEASON, metric='shots', limit=5)
team_shots_table = pd.DataFrame(team_shots)
display(team_shots_table)
assert team_shots_table['shots'].tolist() == sorted(team_shots_table['shots'], reverse=True)
print('Ranking checks passed: both team tables are sorted descending.')

## 9. Pipeline status

Show the latest refresh command, status, error (if any), and row counts.

In [ ]:
# Inspect the latest pipeline run and its refresh row counts.

pipeline = get_json('/pipeline/status')
pipeline_status = {key: value for key, value in pipeline.items() if key != 'row_counts'}
display(pd.Series(pipeline_status, name='value').to_frame())
row_counts = pipeline.get('row_counts') or {}
display(pd.Series(row_counts, name='count').to_frame())
assert pipeline['status'] in {'succeeded', 'running'}
assert pipeline['command'] in {'refresh', 'refresh-as-of', 'backfill'}

## 10. Final regular-season standings — 2025–26

In [ ]:
# Display final regular-season standings for 2025–26.

standings = get_json('/standings', season_id=20252026)
standings_table = pd.DataFrame(standings)
display(standings_table)

## 11. Stanley Cup winners

Playoff champions are not part of regular-season standings, so this is the same explicit completed-season reference list used by the shell test.

In [ ]:
# Display the completed-season Stanley Cup winner reference table.

champions = pd.DataFrame([
    {'season': '2022-2023', 'winner': 'Vegas Golden Knights', 'abbreviation': 'VGK'},
    {'season': '2023-2024', 'winner': 'Florida Panthers', 'abbreviation': 'FLA'},
    {'season': '2024-2025', 'winner': 'Florida Panthers', 'abbreviation': 'FLA'},
    {'season': '2025-2026', 'winner': 'Carolina Hurricanes', 'abbreviation': 'CAR'},
])
display(champions)

## 12. Connor McDavid goals across the available seasons

In [ ]:
# Compare Connor McDavid goal totals across all configured seasons.

SEASONS = [20222023, 20232024, 20242025, 20252026, 20262027]
mcdavid_rows = []
for season_id in SEASONS:
    season_name = f'{season_id // 10000}-{season_id % 10000:04d}'
    rows = get_json('/players/most-goals', season_id=season_id, limit=100)
    match = next((row for row in rows if row.get('player_id') == 8478402), None)
    mcdavid_rows.append({
        'season': season_name,
        'season_id': season_id,
        'player': match.get('name', 'Connor McDavid') if match else 'Connor McDavid',
        'goals': match.get('goals') if match else None,
        'note': None if match else 'no player-season row returned',
    })
mcdavid_table = pd.DataFrame(mcdavid_rows).sort_values('season_id')
display(mcdavid_table)

## 13. Players who appeared for multiple teams

This is an additional API view retained from the original notebook.

In [ ]:
# Identify players listed with multiple team affiliations.

multi_team = get_json('/players/multi-team', season_id=SEED_SEASON)
multi_team_table = pd.DataFrame(multi_team)
print(f'Players with multiple team affiliations: {len(multi_team_table)}')
display(multi_team_table.head(15))
assert len(multi_team_table) > 0
assert (multi_team_table['team_count'] >= 2).all()
assert (multi_team_table['team_changes'] == multi_team_table['team_count'] - 1).all()

## 14. Active Tampa Bay roster

Show the latest roster snapshot as a table.

In [ ]:
# Display the current Tampa Bay roster snapshot.

tampa_roster = get_json('/rosters/current/TBL')
roster_table = pd.DataFrame(tampa_roster)
display(roster_table.head(15))
assert len(roster_table) > 0
assert {'player_id', 'team', 'position', 'snapshot_date'}.issubset(roster_table.columns)
assert set(roster_table['team']) == {'TBL'}

## 15. Season and data-volume summary

In [ ]:
# Summarize the latest refresh status and updated row counts.

summary = {
    'pipeline_status': pipeline['status'],
    'last_command': pipeline['command'],
    'season_checked': row_counts.get('season_id'),
    'dates_checked': row_counts.get('dates_checked'),
    'games_updated': row_counts.get('games'),
    'player_game_rows_updated': row_counts.get('player_game_stats'),
    'team_game_rows_updated': row_counts.get('team_game_stats'),
    'standings_snapshots': row_counts.get('standings_snapshots'),
    'roster_snapshots': row_counts.get('roster_snapshots'),
}
display(pd.Series(summary, name='value').to_frame())

## 16. Seed-file quality cross-check

The provided CSV is the required starting dataset. This cell checks the documented properties without using it to answer the API questions: 951 rows, one season, unique player IDs, and the known McDavid value.

In [ ]:
# Cross-check the supplied seed CSV against its documented properties.

candidates = [Path("data/data_dump.csv"), Path.cwd().parent / "data/data_dump.csv"]
csv_path = next((candidate for candidate in candidates if candidate.exists()), None)
if csv_path is None:
    raise FileNotFoundError("Could not locate data/data_dump.csv from the notebook or repository parent directory.")
seed = pd.read_csv(csv_path, keep_default_na=False)
quality = {
    "path": str(csv_path),
    "rows": len(seed),
    "columns": len(seed.columns),
    "unique_player_ids": seed["playerId"].nunique(),
    "seasons": sorted(seed["Season"].unique().tolist()),
    "mcdavid_goals": int(seed.loc[seed["Name"] == "Connor McDavid", "G"].iloc[0]),
    "multi_team_rows": int(seed["Team"].str.contains(",").sum()),
}
pd.Series(quality)

## 17. Robustness checks

A malformed metric should produce a client error rather than silently returning a misleading result; an unknown route should produce a 404.

In [ ]:
# Verify that invalid metrics and unknown routes fail explicitly.

bad_metric = requests.get(
    f"{BASE_URL}/teams/rankings",
    params={"season_id": SEED_SEASON, "metric": "invalid"},
    timeout=REQUEST_TIMEOUT,
)
missing_route = requests.get(f"{BASE_URL}/does-not-exist", timeout=REQUEST_TIMEOUT)
print({
    "invalid_metric_status": bad_metric.status_code,
    "unknown_route_status": missing_route.status_code,
})
assert bad_metric.status_code == 400
assert missing_route.status_code == 404
print("Robustness checks passed.")

## 18. Reproducibility and handoff commands

These are the commands another engineer can run from a clean checkout. The notebook is evidence on top of the reproducible service, not a replacement for Docker Compose.

In [ ]:
# Print reproducibility commands for local startup, testing, refresh, and shutdown.

print("""# Start the full stack
./start.sh

# Run this notebook from the repository root
jupyter lab

# Or run the automated tests inside the API image
docker compose run --rm api python -m pytest -q

# Run the daily correction-window refresh
./refresh.sh

# Stop while preserving the PostgreSQL volume
./stop.sh""")

---

# CONCLUSION

The demonstrated vertical slice is complete: Docker Compose starts PostgreSQL and FastAPI, the seed CSV is loaded into normalized tables, the API answers every requested analyst question, refreshes are observable and repeatable, and invalid requests fail explicitly. The main operational handoff is the daily refresh command/workflow; local Docker Compose remains the reproducible way for another engineer to run the system.

---

# GALLERY (Tests, Verifications, Other Slop to Show Off)
Screenshots of various features, steps, and test runs I screenshotted and ran repeately (like for data/endpoint verification tasks for example whenever I changed or added something to the backend) throughout my development process. Helps show the workflow and thought process I had throughout my planning as this assessment was fairly open ended. 

***Includes for example screenshots of automated or potential future automated sections if further built out. Such as the GitHub Action I already have working daily to rescrape the past days games and update the Supabase/Postgres dataset in https://github.com/bestisblessed/nhl-ai with the daily update success or failure email screenshot from the run this morning.***

In [ ]:
# List the files available for the notebook evidence gallery.

!ls FINAL-DELIVERABLE-TESTS-AND-EXAMPLES

In [ ]:
#  Iterate and display all final images in specified directory 

IMAGE_DIR = Path("FINAL-DELIVERABLE-TESTS-AND-EXAMPLES")
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".gif", ".webp", ".bmp"}
MAX_DISPLAY_WIDTH = 1400

def readable_caption(path: Path) -> str:
    name = path.stem.replace("_", " ").replace("-", " ")
    name = re.sub(r"([A-Z]+)([A-Z][a-z])", r"\1 \2", name)
    name = re.sub(r"([a-z0-9])([A-Z])", r"\1 \2", name)
    return re.sub(r"\s+", " ", name).strip()

image_paths = sorted(
    path for path in IMAGE_DIR.iterdir()
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
)

if not image_paths:
    print(f"No supported images found in: {IMAGE_DIR.resolve()}")
else:
    for image_path in image_paths:
        caption = readable_caption(image_path)
        display(
            NotebookImage(
                filename=str(image_path),
                width=MAX_DISPLAY_WIDTH,
                retina=True,
            )
        )
        display(
            HTML(
                f"""
                <div style="
                    text-align: center;
                    color: #687985;
                    font-size: 11px;
                    font-style: italic;
                    margin: -2px 0 20px 0;
                    line-height: 1.2;
                ">
                    {caption}
                </div>
                """
            )
        )